# 05 · Resultados sintéticos: limites de independência e critérios

A linha sintética é **complementar e exploratória**.

* Os 22 cenários **compartilham estrutura** (mesma rede, mesma frota, mesma geração de pedidos); variantes que só alteram parâmetros do planejador D+1 abandonado não afetam o despacho em lotes. Logo, "22 de 22" **não** são 22 confirmações independentes.
* As sementes finais 20001–20040 foram **examinadas antes** da decisão de adotar o método.
* **Critério de serviço sintético**: razão média pareada de atendidos ≥ 0,995 **e** razão das médias de P90 ≤ 1,05. O critério **E2 histórico** é mais rígido: atendidos(C) ≥ atendidos(referência), sem tolerância. Os critérios não foram alterados retroativamente.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "execute o notebook dentro do repositório"
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
SEED = 20261108
np.random.seed(SEED)

In [2]:
path = ROOT / "reports/synthetic/dispatch_evaluation/analise/resumo_contrastes.csv"
d = pd.read_csv(path)
c = d[(d.politica == "D_lote60s_hungaro") & (d.referencia == "A")].reset_index(drop=True)
assert len(c) == 22
print("C × A: cenários com IC95% < 0:", int((c.ic95_sup < 0).sum()), "de", len(c))
print(f"mediana da variação do km vazio: {c.d_pct.median():.2f}%  | mín {c.d_pct.min():.2f}% | máx {c.d_pct.max():.2f}%")
assert (c.ic95_sup < 0).all() and round(c.d_pct.median(), 1) == -8.4

C × A: cenários com IC95% < 0: 22 de 22
mediana da variação do km vazio: -8.36%  | mín -14.04% | máx -2.15%


## Os cenários principais (confirmatórios) com Holm

In [3]:
main = c[c.cenario.isin(["S1", "S2", "S3", "S4", "S5"])][
    [
        "cenario",
        "d_pct",
        "ic95_inf",
        "ic95_sup",
        "p_boot",
        "p_holm_confirmatorio",
        "razao_atendidos",
        "razao_p90",
        "servico_preservado",
        "classe",
    ]
]
main.round(4)

   cenario   d_pct  ic95_inf  ic95_sup  p_boot  p_holm_confirmatorio  razao_atendidos  razao_p90  servico_preservado                          classe
0       S1 -2.1477  -65.6412  -33.9744  0.0001                 0.002           0.9984     1.0003                True  reducao_com_servico_preservado
1       S2 -9.1439 -235.0376 -190.1683  0.0001                 0.002           1.0098     0.9334                True  reducao_com_servico_preservado
10      S3 -7.9039 -189.4043 -148.7914  0.0001                 0.002           1.0187     0.9179                True  reducao_com_servico_preservado
18      S4 -7.8712 -212.4067 -179.3818  0.0001                 0.002           1.0153     0.9452                True  reducao_com_servico_preservado
20      S5 -8.4108 -234.2916 -176.3697  0.0001                 0.002           1.0131     0.9410                True  reducao_com_servico_preservado

## Onde o critério sintético é mais frouxo que o E2 histórico

In [4]:
loose = c[c.razao_atendidos < 1.0][["cenario", "razao_atendidos", "razao_p90", "servico_preservado"]]
print("cenários com razão de atendidos < 1 (C atende, em média, menos que A):", len(loose))
loose

cenários com razão de atendidos < 1 (C atende, em média, menos que A): 2


              cenario  razao_atendidos  razao_p90  servico_preservado
0                  S1         0.998392   1.000335                True
3  S2_frota_abundante         0.996264   0.996350                True

In [5]:
from IPython.display import Image, display

from tcc_frota.visualization import figures

out = ROOT / "reports/figures"
figures.plot_synthetic_summary(d, out)
display(Image(filename=str(out / "synthetic_contrasts.png")))

<IPython.core.display.Image object>

**Leitura.** O sentido da diferença (C usa menos km vazio que A) aparece em todos os cenários, mas os cenários não são independentes, as sementes finais já tinham sido vistas, e a simulação não mede efeito operacional. A evidência sintética **complementa** a histórica; não a substitui.